In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
def vgg_block(num_convs, out_channels): # 输入参数：卷积层数，输出通道数
    layers = []
    for _ in range(num_convs): # 循环 num_convs 次
        layers.append(nn.LazyConv2d(out_channels, kernel_size=3, padding=1))
        # 卷积层，输出通道数为 out_channels，卷积核大小为 3，填充为 1
        layers.append(nn.ReLU()) # 激活函数
    layers.append(nn.MaxPool2d(kernel_size=2,stride=2)) # 池化层，大小为 2，步长为 2
    return nn.Sequential(*layers) # *layers 解包，Sequential 作用是将多个层组合成一个网络

In [3]:
class VGG(d2l.Classifier):
    def __init__(self, arch, lr=0.1, num_classes=10): # 输入参数：网络架构，学习率，分类数目
        super().__init__()
        self.save_hyperparameters()
        conv_blks = [] # 卷积块列表
        for (num_convs, out_channels) in arch:
            conv_blks.append(vgg_block(num_convs, out_channels))
        self.net = nn.Sequential(
            *conv_blks, nn.Flatten(), # 前面的块展平后再全连接
            nn.LazyLinear(4096), nn.ReLU(), nn.Dropout(0.5),
            nn.LazyLinear(4096), nn.ReLU(), nn.Dropout(0.5),
            nn.LazyLinear(num_classes))
        self.net.apply(d2l.init_cnn) # 对卷积层中的权重进行初始化

In [4]:
VGG(arch=((1, 64), (1, 128), (2, 256), (2, 512), (2, 512))).layer_summary((1, 1, 224, 224)) # layer_summary 这里创建一个输入模拟进行输出，显示各层输出形状

Sequential output shape:	 torch.Size([1, 64, 112, 112])
Sequential output shape:	 torch.Size([1, 128, 56, 56])
Sequential output shape:	 torch.Size([1, 256, 28, 28])
Sequential output shape:	 torch.Size([1, 512, 14, 14])
Sequential output shape:	 torch.Size([1, 512, 7, 7])
Flatten output shape:	 torch.Size([1, 25088])
Linear output shape:	 torch.Size([1, 4096])
ReLU output shape:	 torch.Size([1, 4096])
Dropout output shape:	 torch.Size([1, 4096])
Linear output shape:	 torch.Size([1, 4096])
ReLU output shape:	 torch.Size([1, 4096])
Dropout output shape:	 torch.Size([1, 4096])
Linear output shape:	 torch.Size([1, 10])


In [ ]:
model = VGG(arch=((1, 16), (1, 32), (2, 64), (2, 128), (2, 128)), lr=0.01) # 模型
trainer = d2l.Trainer(max_epochs=10, num_gpus=1) # 训练器
data = d2l.FashionMNIST(batch_size=128, resize=(224, 224)) # 数据
model.apply_init([next(iter(data.get_dataloader(True)))[0]], d2l.init_cnn) # 初始化卷积层
trainer.fit(model, data) # 训练
# 本机算力不足，没法跑